In [1]:
from dotenv import load_dotenv
from langchain_community.document_loaders import UnstructuredMarkdownLoader
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings
import os
import yaml
from pathlib import Path
import chromadb
import shutil
import hashlib
import numpy as np
from openai import OpenAIError

# Пути и параметры
md_path = Path('c:/Users/Alkor/gd/news_rss_md_rts_21-00')
chromadb_path = './chroma_db_chatgpt_graph_rts_21-00'
model_name = "text-embedding-3-small"  # Модель OpenAI для эмбеддингов
batch_size = 5  # Размер батча для добавления документов

# Загрузка ключа API ChatGPT из .env файла
load_dotenv(override=True)
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY', 'your-key-if-not-using-env')

# Кастомная функция эмбеддингов для ChromaDB
class OpenAIEmbeddingFunction:
    def __init__(self, api_key, model_name):
        self.embeddings = OpenAIEmbeddings(api_key=api_key, model=model_name)
    
    def __call__(self, input):
        return self.embeddings.embed_documents(input)

def get_folder_size(folder_path):
    total_size = 0
    for dirpath, _, filenames in os.walk(folder_path):
        for f in filenames:
            fp = os.path.join(dirpath, f)
            total_size += os.path.getsize(fp)
    return total_size / (1024 * 1024)  # Размер в МБ

def load_markdown_files(directory):
    documents = []
    for file_path in list(directory.glob("**/*.md")):
        # Чтение содержимого файла
        with open(file_path, 'r', encoding='utf-8') as file:
            content = file.read()
        
        # Разделение метаданных и текста
        if content.startswith('---'):
            parts = content.split('---', 2)
            if len(parts) >= 3:
                metadata_yaml = parts[1].strip()
                text_content = parts[2].strip()
                # Парсинг метаданных
                metadata = yaml.safe_load(metadata_yaml) or {}
                # Преобразование метаданных в строки
                metadata_str = {
                    "next_bar": str(metadata.get("next_bar", "unknown")),
                    "date_min": str(metadata.get("date_min", "unknown")),
                    "date_max": str(metadata.get("date_max", "unknown")),
                    "source": file_path.name,
                    "date": file_path.stem
                }
                # Создание объекта Document
                doc = Document(
                    page_content=text_content,
                    metadata=metadata_str
                )
                documents.append(doc)
            else:
                # Если нет метаданных, добавляем unknown
                doc = Document(
                    page_content=content,
                    metadata={
                        "next_bar": "unknown",
                        "date_min": "unknown",
                        "date_max": "unknown",
                        "source": file_path.name,
                        "date": file_path.stem
                    }
                )
                documents.append(doc)
        else:
            # Если нет секции метаданных
            doc = Document(
                page_content=content,
                metadata={
                    "next_bar": "unknown",
                    "date_min": "unknown",
                    "date_max": "unknown",
                    "source": file_path.name,
                    "date": file_path.stem
                }
            )
            documents.append(doc)
    return documents

# Функция для добавления документов батчами
def add_in_batches(collection, ids, documents, metadatas, batch_size):
    for i in range(0, len(documents), batch_size):
        batch_ids = ids[i:i + batch_size]
        batch_docs = documents[i:i + batch_size]
        batch_metas = metadatas[i:i + batch_size]
        try:
            print(f"Добавление батча {i // batch_size + 1} с {len(batch_docs)} документами")
            collection.add(ids=batch_ids, documents=batch_docs, metadatas=batch_metas)
            print(f"Батч {i // batch_size + 1} успешно добавлен")
        except OpenAIError as e:
            print(f"Ошибка OpenAI API при добавлении батча {i // batch_size + 1}: {e}")
            raise
        except Exception as e:
            print(f"Общая ошибка при добавлении батча {i // batch_size + 1}: {e}")
            raise

# Удаление папки chroma_db, если она существует
if os.path.exists(chromadb_path):
    print(f"Размер папки {chromadb_path} до удаления: {get_folder_size(chromadb_path):.2f} МБ")
    shutil.rmtree(chromadb_path)
    print(f"Папка {chromadb_path} удалена.")

# Инициализация клиента ChromaDB
client = chromadb.PersistentClient(path=chromadb_path)

# Создание функции эмбеддингов для OpenAI
ef = OpenAIEmbeddingFunction(
    api_key=OPENAI_API_KEY,
    model_name=model_name
)

# Создание коллекции
collection = client.create_collection(name="news_collection", embedding_function=ef)

# Загрузка Markdown-файлов
documents = load_markdown_files(md_path)

# Проверка на пустую папку
if not documents:
    print("Не найдено Markdown-файлов в указанной директории.")
    exit(1)
else:
    print(f"Загружено {len(documents)} Markdown-файлов из {md_path}")
    print(f"Документы даты: {set(doc.metadata['date'] for doc in documents)}")
    print(f"Направление следующего бара: {set(doc.metadata['next_bar'] for doc in documents)}")
    print(f"Минимальные даты: {set(doc.metadata['date_min'] for doc in documents)}")
    print(f"Максимальные даты: {set(doc.metadata['date_max'] for doc in documents)}")

# Подготовка данных для ChromaDB
doc_texts = [doc.page_content for doc in documents]
doc_ids = [hashlib.md5(doc.page_content.encode()).hexdigest() for doc in documents]
doc_metadatas = [doc.metadata for doc in documents]

# Добавление в коллекцию батчами
try:
    add_in_batches(collection, doc_ids, doc_texts, doc_metadatas, batch_size)
    print("Все документы успешно добавлены.")
except Exception as e:
    print(f"Ошибка при добавлении документов: {e}")
    exit(1)

# # Пример поиска с фильтрацией по метаданным
# query = "Новости о Tesla"
# results = collection.query(
#     query_texts=[query],
#     n_results=3,
#     where={"next_bar": "up"}  # Фильтрация по next_bar
#     # where={"next_bar": "up", "date": {"$eq": "2025-07-24"}}
# )
# print(results)

Размер папки ./chroma_db_chatgpt_graph_rts_21-00 до удаления: 75.06 МБ
Папка ./chroma_db_chatgpt_graph_rts_21-00 удалена.
Загружено 30 Markdown-файлов из c:\Users\Alkor\gd\news_rss_md_rts_21-00
Документы даты: {'2025-07-23', '2025-08-05', '2025-08-01', '2025-07-14', '2025-06-27', '2025-07-17', '2025-06-30', '2025-07-29', '2025-07-30', '2025-07-11', '2025-06-25', '2025-07-21', '2025-07-02', '2025-07-18', '2025-07-28', '2025-07-04', '2025-08-04', '2025-07-31', '2025-07-09', '2025-07-01', '2025-07-08', '2025-07-10', '2025-06-26', '2025-07-22', '2025-07-24', '2025-07-03', '2025-07-07', '2025-07-16', '2025-07-25', '2025-07-15'}
Направление следующего бара: {'None', 'up', 'down'}
Минимальные даты: {'2025-07-17 18:00:00', '2025-07-28 18:00:00', '2025-06-27 18:00:00', '2025-07-22 18:00:00', '2025-06-24 18:00:00', '2025-07-08 18:00:00', '2025-07-29 18:00:00', '2025-07-31 18:00:00', '2025-07-07 18:00:00', '2025-07-03 18:00:00', '2025-07-25 18:00:00', '2025-07-11 18:00:00', '2025-06-30 18:00:00',

In [2]:
from sklearn.manifold import TSNE
import numpy as np
import plotly.graph_objects as go
from datetime import datetime

# Пути и параметры
output_dir = Path('C:/Users/Alkor/gd/rss_news_predict_quote/OpenAI_predict_img_rts_21-00')

# Создаем директорию для сохранения изображений, если она не существует
output_dir.mkdir(parents=True, exist_ok=True)

# Теперь мы можем визуализировать векторы с помощью t-SNE
# t-SNE - это метод, который позволяет визуализировать высокоразмерные данные
result = collection.get(include=['embeddings', 'documents', 'metadatas'])
vectors = np.array(result['embeddings'])
documents = result['documents']
metadatas = result['metadatas']
doc_types = [metadata['next_bar'] for metadata in metadatas]
doc_dates = [metadata['date'] for metadata in metadatas]
doc_date_mins = [metadata['date_min'] for metadata in metadatas]
colors = [['blue', 'red', 'black'][['up', 'down', 'None'].index(t)] for t in doc_types]


# Нам, людям, проще визуализировать объекты в 2D!
# Уменьшите размерность векторов до 2D, используя t-SNE
# (t-распределенное стохастическое вложение соседей)
tsne = TSNE(n_components=2, random_state=42, perplexity=3)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 2D scatter plot
fig = go.Figure(data=[go.Scatter(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Date: {dt}    Date Min: {dm}<br>Text: {d[:100]}..." 
          for t, dt, dm, d in zip(doc_types, doc_dates, doc_date_mins, documents)],
    hoverinfo='text'
)])

# Всталяем дату
# graph_date = datetime.now().strftime('%Y-%m-%d %HH:%MM:%SS')
graph_date = '2025-08-05'  # Можно заменить на динамическое значение, если нужно

fig.update_layout(
    title=f'2D Chroma Vector Store Ollama (bge-m3) {graph_date}',
    xaxis_title='x',
    yaxis_title='y',
    width=800,
    height=600,
    margin=dict(r=20, b=10, l=10, t=40)
)

# Сохранение графика в файл
output_file = output_dir / f"{graph_date}.png"
fig.write_image(output_file)

fig.show()

In [3]:
# Let's try 3D!
tsne = TSNE(n_components=3, random_state=42, perplexity=3)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 3D scatter plot
fig = go.Figure(data=[go.Scatter3d(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    z=reduced_vectors[:, 2],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(
    title='3D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='z'),
    width=900,
    height=700,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()

In [4]:
# Косинусное сходство
def cosine_similarity(vec1, vec2):
    vec1 = np.array(vec1)
    vec2 = np.array(vec2)
    dot_product = np.dot(vec1, vec2)
    norm1 = np.linalg.norm(vec1)
    norm2 = np.linalg.norm(vec2)
    if norm1 == 0 or norm2 == 0:
        return 0.0
    return dot_product / (norm1 * norm2)

# Сравнение векторов
def compare_vectors_avg():
    # Получение документов с next_bar="None"
    none_results = collection.query(
        query_texts=[""],  # Пустой запрос, используем фильтр
        n_results=1000,  # Достаточно большой лимит
        where={"next_bar": "None"}
    )
    
    if not none_results['ids'][0]:
        print("Документы с next_bar='None' не найдены.")
        return
    
    none_id = none_results['ids'][0][0]
    none_embedding = collection.get(ids=[none_id], include=["embeddings"])["embeddings"][0]
    
    # Получение документов с next_bar="up"
    up_results = collection.query(
        query_texts=[""],
        n_results=1000,
        where={"next_bar": "up"}
    )
    
    # Получение документов с next_bar="down"
    down_results = collection.query(
        query_texts=[""],
        n_results=1000,
        where={"next_bar": "down"}
    )
    
    # Проверка наличия документов
    if not up_results['ids'][0]:
        print("Документы с next_bar='up' не найдены.")
        return
    if not down_results['ids'][0]:
        print("Документы с next_bar='down' не найдены.")
        return
    
    # Вычисление среднего сходства для up
    up_embeddings = collection.get(ids=up_results['ids'][0], include=["embeddings"])["embeddings"]
    up_similarities = [cosine_similarity(none_embedding, emb) for emb in up_embeddings]
    avg_up_similarity = np.mean(up_similarities) * 100  # В процентах
    
    # Вычисление среднего сходства для down
    down_embeddings = collection.get(ids=down_results['ids'][0], include=["embeddings"])["embeddings"]
    down_similarities = [cosine_similarity(none_embedding, emb) for emb in down_embeddings]
    avg_down_similarity = np.mean(down_similarities) * 100  # В процентах
    
    # Вывод результатов
    print(f"Среднее сходство с категорией 'up': {avg_up_similarity:.2f}%")
    print(f"Среднее сходство с категорией 'down': {avg_down_similarity:.2f}%")
    
    # Определение ближайшей категории
    if avg_up_similarity > avg_down_similarity:
        print("Документ с next_bar='None' ближе к категории 'up'.")
    elif avg_down_similarity > avg_up_similarity:
        print("Документ с next_bar='None' ближе к категории 'down'.")
    else:
        print("Документ с next_bar='None' имеет одинаковое сходство с категориями 'up' и 'down'.")

# Выполнение сравнения
compare_vectors_avg()

Среднее сходство с категорией 'up': 92.70%
Среднее сходство с категорией 'down': 91.37%
Документ с next_bar='None' ближе к категории 'up'.


In [5]:
def compare_vectors():
    # Получение документа с next_bar="None"
    none_results = collection.query(
        query_texts=[""],  # Пустой запрос, используем фильтр
        n_results=1,  # Только один документ
        where={"next_bar": "None"}
    )
    
    if not none_results['ids'][0]:
        print("Документ с next_bar='None' не найден.")
        return
    
    none_id = none_results['ids'][0][0]
    none_embedding = collection.get(ids=[none_id], include=["embeddings"])["embeddings"][0]
    
    # Получение всех документов, кроме документа с next_bar="None"
    all_results = collection.query(
        query_texts=[""],
        n_results=1000,  # Достаточно большой лимит
        where={"next_bar": {"$ne": "None"}}  # Исключаем next_bar="None"
    )
    
    if not all_results['ids'][0]:
        print("Другие документы не найдены.")
        return
    
    # Получение векторов и метаданных для всех документов
    all_ids = all_results['ids'][0]
    all_embeddings = collection.get(ids=all_ids, include=["embeddings", "metadatas"])["embeddings"]
    all_metadatas = collection.get(ids=all_ids, include=["metadatas"])["metadatas"]
    
    # Вычисление косинусного сходства
    similarities = [
        (cosine_similarity(none_embedding, emb) * 100, meta, doc_id) 
        for emb, meta, doc_id in zip(all_embeddings, all_metadatas, all_ids)
    ]
    
    # Сортировка по убыванию сходства
    similarities.sort(key=lambda x: x[0], reverse=True)
    
    # Вывод метаданных и процента сходства для трех ближайших документов
    print("Три ближайших документа по сходству с документом next_bar='None':")
    for i, (similarity, metadata, doc_id) in enumerate(similarities[:3], 1):
        print(f"\nДокумент {i}:")
        print(f"Процент сходства: {similarity:.2f}%")
        print("Метаданные:")
        for key, value in metadata.items():
            print(f"  {key}: {value}")

# Выполнение сравнения векторов
compare_vectors()

Три ближайших документа по сходству с документом next_bar='None':

Документ 1:
Процент сходства: 98.13%
Метаданные:
  date_min: 2025-07-29 18:00:00
  next_bar: up
  date_max: 2025-07-30 18:00:00
  date: 2025-07-30
  source: 2025-07-30.md

Документ 2:
Процент сходства: 97.82%
Метаданные:
  next_bar: down
  date_max: 2025-07-31 18:00:00
  date_min: 2025-07-30 18:00:00
  date: 2025-07-31
  source: 2025-07-31.md

Документ 3:
Процент сходства: 97.26%
Метаданные:
  date: 2025-07-24
  date_min: 2025-07-23 18:00:00
  source: 2025-07-24.md
  next_bar: down
  date_max: 2025-07-24 18:00:00
